In [8]:
from langgraph.graph import StateGraph , START , END
from pydantic import BaseModel , Field
from typing import TypedDict , Literal
import os 
import operator
from dotenv import load_dotenv


In [5]:
load_dotenv()

False

In [9]:
class SentimentSchema(BaseModel):
    sentiment : Literal["positive" , "negative"] = Field(description = 'Sentiment of the review')

In [ ]:
structure_model = model.with_structure_output(SentimentSchema)

In [15]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [ ]:
structured_model2 = model.with_structured_output(DiagnosisSchema)

In [10]:
class ReviewState(TypedDict):
    review : str
    sentiment : str
    diagnosis : dict
    response : str
    

In [13]:
def find_sentiment(state:ReviewState):
    prompt = f"For the following review find out the sentiment \n {state['review']}"
    sentiment =structured_model.invoke(prompt)
    return {'sentiment' : sentiment}

In [11]:
def check_sentiment(state: ReviewState) -> Literal["Positive_response" ,"run_diagnosis"]:
    if state["sentiment"] == "positive":
        return 'positive_response'
    else:
        return 'run_diagnosis'

In [ ]:
def positive_response(state:ReviewState):
    prompt = f"""Write a warm thank you message in response to this review:
     \n \n "{state['review']}" \n 
     Also , kindly ask the user to leave feedback on our webiste 
     """
     response = model.invoke(prompt).content
     return {'response' : response}
 


In [ ]:
def run_diagnosis(state: ReviewState):

    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
"""
    response = structured_model2.invoke(prompt)

    return {'diagnosis': response.model_dump()}

In [ ]:
def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = model.invoke(prompt).content

    return {'response': response}

In [18]:
graph  = StateGraph(ReviewState)
    

graph.add_node('find_sentiment' , find_sentiment)
graph.add_node('positive_response ' , positive_response)
graph.add_node('run_diagnosis' , run_diagnosis)
graph.add_node('negative_response' , negative_response)



graph.add_edge(START , 'find_sentiment')
graph.add_conditional_edge('find_sentiment' , check_sentiment)
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis' , 'negative_response')
graph.add_edge('negative_response' , END)
# graph.add_edge('find_sentiment' , END)

graph.compile()

NameError: name 'positive_response' is not defined